# Retrieval-Augmented Generation (RAG)

In [1]:
import chromadb
import dotenv
from pathlib import Path
from agents import Agent, Runner, function_tool, trace

dotenv.load_dotenv()

True

Create a static calorie table that we can use as a tool:

In [ ]:
# We populated the RAG with the data from the data/calories.csv file in
# the rag_setup.ipynb notebook
# connection to sqlite database
chroma_client = chromadb.PersistentClient("../chroma")
nutrition_db = chroma_client.get_collection(name="nutrition_db")

In [ ]:
# example query command
results = nutrition_db.query(query_texts=["banana"], n_results=2)
for i, doc in enumerate(results["documents"][0]):
    print(sorted(results["metadatas"][0][i].items()))
    print(doc)
    print("\n")

[('calories_per_100g', 89.0), ('food_category', 'fruits'), ('food_item', 'banana'), ('keywords', 'banana_fruits'), ('kj_per_100g', 374.0), ('serving_info', '100g')]
Food: Banana
        Category: Fruits
        Nutritional Information:
        - Calories: 89 per 100g
        - Energy: 374 kJ per 100g
        - Serving size reference: 100g

        This is a fruits food item that provides 89 calories per 100 grams.


[('calories_per_100g', 50.0), ('food_category', '(fruit)juices'), ('food_item', 'banana juice'), ('keywords', 'banana_juice_(fruit)juices'), ('kj_per_100g', 210.0), ('serving_info', '100ml')]
Food: Banana Juice
        Category: (Fruit)Juices
        Nutritional Information:
        - Calories: 50 per 100g
        - Energy: 210 kJ per 100g
        - Serving size reference: 100ml

        This is a (fruit)juices food item that provides 50 calories per 100 grams.




In [ ]:
# everything put together. Uses query call from above
#@function_tool
def calorie_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up calorie information for specific food items, but not for meals.

    Args:
        query: The food item to look up.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the nutrition information.
    """

    results = nutrition_db.query(query_texts=[query], n_results=max_results)

    # "documents" are retrieved documents from the query
    if not results["documents"][0]:
        return f"No nutrition information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        food_item = metadata["food_item"].title()
        calories = metadata["calories_per_100g"]
        category = metadata["food_category"].title()

        formatted_results.append(
            f"{food_item} ({category}): {calories} calories per 100g"
        )

    return "Nutrition Information:\n" + "\n".join(formatted_results)
    #return results

Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `calorie_lookup_tool` function_

In [ ]:
# calorie_lookup_tool('bananas')

'Nutrition Information:\nBanana (Fruits): 89.0 calories per 100g\nBanana Juice ((Fruit)Juices): 50.0 calories per 100g\nBanana Nut Bread (Pastries,Breads&Rolls): 326.0 calories per 100g'

In [27]:
# Test the setup with sample queries
# chroma_client = chromadb.PersistentClient("../chroma")
nutrition_qna = chroma_client.get_collection(name="nutrition_qna")

# Test query 1: Search for malnutrition symptoms
print("=== Query: pregnancy ===")
results = nutrition_qna.query(query_texts=["pregnancy"], n_results=3)
for i, doc in enumerate(results["documents"][0]):
    print(f"Result {i+1}:")
    print(f"Answer: {results['documents'][0]}")
    print("\n")

print("\n" + "=" * 50 + "\n")

=== Query: pregnancy ===
Result 1:
Answer: ['Question: What are some possible physical changes that a pregnant woman could experience in the middle months of gestation?\n        Answer: During weeks 13-27, you may see an increase in weight and feel more hunger. Backaches might occur frequently, along with leg cramps and heartburn.\n\n        This Q&A pair provides information about nutrition and health topics.', 'Question: What health issues can make a pregnancy more challenging?\n        Answer: There are several medical conditions that can potentially increase the risk associated with Pregnancy. These include Anemia during this stage, Hypertensive Disorders related to gestation, Diabetes Mellitus coexisting with Pregnancy, Obesity during pregnancy, as well as Adolescent or Teenage pregnancies.\n\n        This Q&A pair provides information about nutrition and health topics.', "Question: What are some hormonal changes that take place in a mother's body during pregnancy?\n        Answer

In [41]:
# add tool to get Q&A pairs
@function_tool
def QA_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up general nutrition information.

    Args:
        query: The question to look up.
        max_results: The maximum number of results to return.

    Returns:
        A string containing a question and an answer. The question gives the context, while the answer gives the nutritional insights.
    """

    results = nutrition_qna.query(query_texts=[query], n_results=max_results)

    # "documents" are retrieved documents from the query
    print(results["documents"][0])
    if not results["documents"][0]:
        return f"No Q&A information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        # print(f"Result {i+1}:")
        # print(f"Answer: {results['documents'][0]}")
        # print("\n")

        formatted_results.append(
            f"Answer: {results['documents'][0]}"
        )

    return formatted_results
    #return results


In [33]:
QA_lookup_tool("pregnancy")

['Question: What are some possible physical changes that a pregnant woman could experience in the middle months of gestation?\n        Answer: During weeks 13-27, you may see an increase in weight and feel more hunger. Backaches might occur frequently, along with leg cramps and heartburn.\n\n        This Q&A pair provides information about nutrition and health topics.', 'Question: What health issues can make a pregnancy more challenging?\n        Answer: There are several medical conditions that can potentially increase the risk associated with Pregnancy. These include Anemia during this stage, Hypertensive Disorders related to gestation, Diabetes Mellitus coexisting with Pregnancy, Obesity during pregnancy, as well as Adolescent or Teenage pregnancies.\n\n        This Q&A pair provides information about nutrition and health topics.', "Question: What are some hormonal changes that take place in a mother's body during pregnancy?\n        Answer: During pregnancy, there is an increase in

UnboundLocalError: cannot access local variable 'i' where it is not associated with a value

In [44]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    If you need to look up calorie information, use the calorie_lookup_tool.
    If you need to lookup general nutrition information, use the QA_lookup_tool. 
    Use the QA_lookup_tool to see if you can find information about the specific question first.
    """,
    tools=[calorie_lookup_tool, QA_lookup_tool]
    ,
)

In [45]:
with trace("Nutrition Assistant with RAG"):
    result = await Runner.run(
        calorie_agent,
        "Should a pregnant woman drink alcohol?",
    )
    print(result.final_output)

["Question: Can a mother-to-be safely consume specific nourishing fare she desires but are usually not recommended during pregnancy?\n        Answer: As long as the cravings don't interfere with good eating habits and pose no health hazards, consuming nutritious foods like amla or raw tamarind is fine. These items contain vitamin C, which can be advantageous for pregnant women.\n\n        This Q&A pair provides information about nutrition and health topics.", 'Question: Which foods should pregnant women avoid?\n        Answer: Avoiding raw or undercooked meats, unpasteurized dairy products, certain types of fish high in mercury and deli meat can reduce the risk of foodborne illnesses for pregnant women.\n\n        This Q&A pair provides information about nutrition and health topics.', "Question: What should a pregnant woman do when she desires specific healthy yet unconventional dishes?\n        Answer: As long as the cravings don't compromise good eating habits and aren't hazardous to